# ScholarLM Demo

End-to-end extraction of entity–attribute–value triples from a scientific PDF.

**Paper:** Peretyatko et al. (2011) — *Classification trees as a tool for predicting cyanobacterial blooms*

The pipeline moves through five stages: OCR → table cleaning → entity identification → attribute/event detection → value extraction and deduplication. Each stage is checkpointed to disk so individual steps can be inspected independently.

In [ ]:
import sys, json, re, base64
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

sys.path.insert(0, "src")
sys.path.insert(0, "experiments")

PAPER       = "classification_trees"
PDF_PATH    = Path(f"data/pond/pdfs/{PAPER}.pdf")
OCR_RAW     = Path(f"data/pond/ocr_output_raw/{PAPER}.txt")
OCR_CLEANED = Path(f"data/pond/ocr_output_cleaned_qwen-3.5-27b/{PAPER}.txt")

# Intermediate extraction outputs are written here.
DEMO_DIR = Path("data/demo")
DEMO_DIR.mkdir(exist_ok=True)


def get_page(path: Path, n: int) -> str:
    """Return the content of page n (0-indexed) from an OCR .txt file."""
    m = re.search(rf'<page number="{n}">(.*?)</page>', path.read_text(), re.DOTALL)
    return m.group(1).strip() if m else f"[page {n} not found]"


pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 60)

---
## The source document

A 16-page journal article reporting limnological measurements for ~50 ponds.

In [ ]:
import fitz  # PyMuPDF

doc = fitz.open(str(PDF_PATH))
pages_html = []
for page in doc:
    pix = page.get_pixmap(matrix=fitz.Matrix(1.5, 1.5))
    b64 = base64.b64encode(pix.tobytes("png")).decode()
    pages_html.append(
        f'<img src="data:image/png;base64,{b64}" '
        'style="display:block;width:100%;margin-bottom:12px;" />'
    )
doc.close()

display(HTML(
    '<div style="height:720px;overflow-y:scroll;border:1px solid #d0d0d0;'
    'padding:12px;background:#f5f5f5;">'
    + "".join(pages_html)
    + "</div>"
))

---
## Step 1: OCR

olmOCR runs page-by-page using a served vLLM instance. Each page is converted to plain text; tables are emitted as HTML markup.

```bash
# Start the olmOCR server (requires GPU), e.g.:
bash experiments/serve_olmocr.sh

# Run OCR for this paper:
python experiments/run_ocr.py --dataset pond --paper-subset classification_trees
```

Output: `data/pond/ocr_output_raw/classification_trees.txt`

Page 4 (PDF p. 4) contains the main data table. Below is the raw OCR output for that page.

In [ ]:
print(get_page(OCR_RAW, 3))

---
## Step 2: Table cleaning

A vision-language model is shown each page image alongside the raw OCR and asked to restructure any HTML tables into a clean, machine-readable format — normalizing column names, filling merged cells, and removing formatting artifacts.

```bash
python experiments/run_table_cleaning.py \
    --dataset pond \
    --model qwen-3.5-27b \
    --paper-subset classification_trees
```

Output: `data/pond/ocr_output_cleaned_gpt_5_mini/classification_trees.txt`

The same page after cleaning:

In [ ]:
print(get_page(OCR_CLEANED, 3))

---
## Extraction pipeline

The full pipeline processes the cleaned OCR text in seven sequential steps. The cell below runs the complete pipeline programmatically, writing all checkpoint files to `data/demo/`.

```bash
python experiments/run_extraction.py \
    --dataset pond \
    --model gemma-3-27b \
    --paper-subset classification_trees \
    --ocr-dir data/pond/ocr_output_cleaned_qwen-3.5-27b \
    --date demo
```

### Step 1 — Entity identification

The model reads the full document and identifies every distinct aquatic ecosystem, recording its name, alternate identifiers, location, and ecosystem type.

Checkpoint: `entities.json`

In [ ]:
with open(DEMO_DIR / "entities.json") as f:
    entities = json.load(f)

want = ["entity_id", "name", "identifiers", "location", "ecosystem"]
df_e = pd.DataFrame(entities)
display(df_e[[c for c in want if c in df_e.columns]])
print(f"\nTotal entities identified: {len(entities)}")

### Step 2 — Attribute detection

For each document the model determines which target attributes (e.g., `chla`, `tp`, `ph`) are reported anywhere in the text.

Checkpoint: `attributes.json`

In [ ]:
with open(DEMO_DIR / "attributes.json") as f:
    attributes = json.load(f)

# Single-paper run: one key (the document index stored as a string)
doc_attrs = next(iter(attributes.values()))
print("Attributes detected in this paper:")
for attr in doc_attrs:
    print(f"  {attr}")

### Step 3 — Measurement event resolution

Steps 3a–b locate the pages where each entity and attribute co-occur (provenance). Step 3c resolves the measurement event context — date and any additional distinguishing details — for each (entity, attribute, page) intersection.

Checkpoint: `events.json` — keys are `doc|entity_id|attribute|page`.

In [ ]:
with open(DEMO_DIR / "events.json") as f:
    events_raw = json.load(f)

print(f"Total (entity × attribute × page) triples: {len(events_raw)}\n")

rows = []
for key, evts in list(events_raw.items())[:12]:
    _, entity_id, attr, page = key.split("|", 3)
    for evt in evts:
        rows.append({"entity_id": entity_id, "attribute": attr, "page": int(page), **evt})

display(pd.DataFrame(rows))

### Steps 4–5 — Value extraction

For each resolved event, the model extracts numeric values from the relevant text passage (step 4) and from the cleaned table on that page (step 5).

Checkpoint: `values.json`

In [ ]:
with open(DEMO_DIR / "values.json") as f:
    values = json.load(f)

want = ["name", "attribute", "value", "units", "page_number", "source"]
df_values = pd.DataFrame([{k: v.get(k) for k in want} for v in values])
display(df_values.head(20))
print(f"\nTotal raw extractions: {len(values)}")

### Steps 6–7 — Standardization and deduplication

Values are converted to standard units (e.g., mg/L → µg/L), then deduplicated across text and table sources.

Checkpoint: `final.json`

In [ ]:
with open(DEMO_DIR / "final.json") as f:
    final = json.load(f)

all_cols = list(final[0].keys()) if final else []
want = ["measurement_id", "name", "identifiers", "attribute", "value", "units", "document_id"]
display_cols = [c for c in want if c in all_cols]

df_final = pd.DataFrame(final)[display_cols]
display(df_final)
print(f"\nTotal deduplicated measurements: {len(final)}")